<a href="https://colab.research.google.com/github/cristian2434/agente-sapu-isy0101/blob/main/notebooks/agente_sapu_IL2_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EP2 · IL2.3 — Planificación y toma de decisiones: Asistente Clínico SAPU
**ISY0101 · Ingeniería de Soluciones con IA** — mismo caso SAPU. Cubre **IE5** (esquemas de planificación con prioridades) y deja las pruebas de **IE7** (decisiones adaptativas).

**Patrón: Plan-and-Execute con re-planificación** (visto en clase en `1B_basic_planning_agent`):
1. **Planificador (LLM)** propone los pasos en JSON, usando solo acciones permitidas.
2. **Validador** (código) rechaza planes con acciones inventadas o dependencias rotas → si falla, usa un plan de respaldo.
3. **Priorizador** ordena los pasos: clasificación ESI → derivación → protocolo → vademécum → dosis.
4. **Ejecutor** corre cada paso con un `estado` compartido y registra una **traza**.
5. **Reglas adaptativas** (tras cada paso): si el caso empeora o falta información, el sistema **cambia el plan** (reprioriza, descarta o escala al médico).

```
Caso ─► Planificador LLM ─► Validador ─► Priorizador ─► Ejecutor ──► Traza
                                 │ (inválido→plan respaldo)    │
                                 └──────────── Reglas adaptativas ◄┘ (evento / fallo → re-planificar)
```
⚠️ Las reglas ESI y los documentos son de **DEMO**; reemplazar por los criterios reales del Manual ESI/MINSAL.

*(Secciones 1–4 reutilizan el notebook IL2.1: configuración, base documental y herramientas.)*

In [ ]:
import sys
if "google.colab" in sys.modules:
    !pip install -q langchain langchain-classic langchain-openai langchain-community faiss-cpu python-dotenv

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
import os
try:
    from google.colab import userdata
    for _k in ("LLM_API_KEY","LLM_BASE_URL","LLM_MODEL"):
        try: os.environ[_k] = userdata.get(_k)
        except Exception: pass
    os.environ.setdefault("LLM_BASE_URL","https://api.mistral.ai/v1")
    os.environ.setdefault("LLM_MODEL","mistral-small-latest")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
llm = ChatOpenAI(base_url=os.getenv("LLM_BASE_URL"), api_key=os.getenv("LLM_API_KEY"),
                 model=os.getenv("LLM_MODEL","mistral-small-latest"), temperature=0)
embeddings = OpenAIEmbeddings(base_url=os.getenv("LLM_BASE_URL"), api_key=os.getenv("LLM_API_KEY"),
                 model="mistral-embed", check_embedding_ctx_length=False)
print("✅ LLM y embeddings configurados")

✅ LLM y embeddings configurados


## 3. Base documental (DEMO)
Cada fragmento lleva metadatos `documento`, `pagina` y `tipo`, para exigir trazabilidad (objetivo específico de la EP1).
Para el repo final: cargar PDFs reales con `PyPDFLoader` + `RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)` y conservar los metadatos.

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

DOCS = [
 Document(page_content="DEMO. Crisis asmática moderada pediátrica: administrar salbutamol inhalado con aerocámara, evaluar respuesta a los 20 minutos. Si no hay mejoría, considerar corticoide sistémico y derivación.",
          metadata={"documento":"Guía Clínica Asma <15 años (DEMO)","pagina":12,"tipo":"guia"}),
 Document(page_content="DEMO. Hidrocortisona de urgencia: dosis ilustrativa 5 mg/kg por dosis, máximo 100 mg por dosis.",
          metadata={"documento":"Vademécum SAPU (DEMO)","pagina":4,"tipo":"vademecum"}),
 Document(page_content="DEMO. Salbutamol: presentación inhalador 100 mcg/puff. Disponible en botiquín de urgencia.",
          metadata={"documento":"Vademécum SAPU (DEMO)","pagina":7,"tipo":"vademecum"}),
 Document(page_content="DEMO. Dolor torácico en adulto: derivación inmediata al hospital base si hay dolor opresivo >20 min, compromiso hemodinámico o alteraciones en ECG. Traslado con soporte avanzado.",
          metadata={"documento":"Protocolo de Traslado y Derivación SAPU (DEMO)","pagina":3,"tipo":"derivacion"}),
 Document(page_content="DEMO. Categorización ESI: nivel 1 requiere intervención inmediata que salve la vida; nivel 2 alto riesgo o confusión; niveles 3-5 según recursos esperados.",
          metadata={"documento":"Manual de Categorización ESI (DEMO)","pagina":2,"tipo":"triage"}),
]
vector_db = FAISS.from_documents(DOCS, embeddings)
print(f"✅ {len(DOCS)} fragmentos indexados")

/tmp/ipykernel_3344/1661974649.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


✅ 5 fragmentos indexados


## 4. Herramientas del agente

In [ ]:
import re, json, datetime, pathlib
from langchain_classic.agents import tool

# Guardrail de la EP1: bloquear datos identificables (RUT) antes de llegar al LLM / archivos
RUT = re.compile(r"\b\d{1,2}\.?\d{3}\.?\d{3}-[\dkK]\b")
def contiene_datos_paciente(texto: str) -> bool:
    return bool(RUT.search(texto))

# ---------- CONSULTA 1: RAG ----------
@tool
def buscar_protocolo(consulta: str) -> str:
    """Busca en guías clínicas, protocolos de derivación y manual ESI del SAPU.
    Devuelve fragmentos con documento y página. Úsala para cualquier pregunta clínica o de derivación."""
    try:
        res = vector_db.similarity_search(consulta, k=3)
        if not res: return "SIN_INFORMACION: no hay fragmentos relevantes."
        return "\n".join(f"[{d.metadata['documento']}, p.{d.metadata['pagina']}] {d.page_content}" for d in res)
    except Exception as e:
        return f"Error en la búsqueda: {e}"

# ---------- CONSULTA 2: vademécum (filtro por metadato) ----------
@tool
def consultar_vademecum(farmaco: str) -> str:
    """Consulta el vademécum del SAPU para un fármaco (dosis por kg, tope máximo, disponibilidad).
    Si el fármaco no aparece, informa que NO está en el botiquín."""
    try:
        res = vector_db.similarity_search(farmaco, k=4, filter={"tipo":"vademecum"})
        hits = [d for d in res if farmaco.lower().split()[0] in d.page_content.lower()]
        if not hits: return f"'{farmaco}' NO figura en el vademécum del SAPU. No inventar datos."
        return "\n".join(f"[{d.metadata['documento']}, p.{d.metadata['pagina']}] {d.page_content}" for d in hits)
    except Exception as e:
        return f"Error en la consulta: {e}"

# ---------- RAZONAMIENTO: cálculo determinista ----------
@tool
def calcular_dosis(peso_kg: float, dosis_mg_por_kg: float, max_mg: float) -> str:
    """Calcula la dosis (mg) = peso_kg * dosis_mg_por_kg, aplicando el tope max_mg.
    Los valores dosis_mg_por_kg y max_mg DEBEN venir de consultar_vademecum, nunca inventarse."""
    if peso_kg <= 0 or dosis_mg_por_kg <= 0 or max_mg <= 0:
        return "Error: peso, dosis y máximo deben ser positivos."
    bruta = peso_kg * dosis_mg_por_kg
    final = min(bruta, max_mg)
    nota = " (se aplicó el tope máximo)" if bruta > max_mg else ""
    return f"Dosis calculada: {final:g} mg{nota}. Requiere validación del médico a cargo."

# ---------- ESCRITURA: ficha de derivación ----------
@tool
def registrar_derivacion(motivo: str, criterio_protocolo: str, prioridad_esi: int) -> str:
    """Guarda una ficha de derivación al hospital base en archivo. NO incluir nombre ni RUT del paciente.
    Parámetros: motivo clínico breve, criterio del protocolo (con fuente) y prioridad ESI (1-5)."""
    if contiene_datos_paciente(motivo + criterio_protocolo):
        return "BLOQUEADO: se detectaron datos identificables (RUT). Reformule sin datos del paciente."
    if prioridad_esi not in (1,2,3,4,5):
        return "Error: prioridad_esi debe estar entre 1 y 5."
    try:
        out = pathlib.Path("derivaciones"); out.mkdir(exist_ok=True)
        ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        f = out / f"derivacion_{ts}.json"
        f.write_text(json.dumps({"fecha":ts,"motivo":motivo,"criterio":criterio_protocolo,
                                 "esi":prioridad_esi,"estado":"pendiente_validacion_medica"},
                                ensure_ascii=False, indent=2), encoding="utf-8")
        return f"Ficha guardada en {f} (pendiente de validación médica)."
    except Exception as e:
        return f"Error al guardar: {e}"

tools = [buscar_protocolo, consultar_vademecum, calcular_dosis, registrar_derivacion]
print("✅ Herramientas:", [t.name for t in tools])

✅ Herramientas: ['buscar_protocolo', 'consultar_vademecum', 'calcular_dosis', 'registrar_derivacion']


## 5. Planificador (LLM) + validador

In [ ]:
import json, re
from langchain_core.prompts import ChatPromptTemplate

ACCIONES = ["clasificar_esi", "consultar_protocolo", "consultar_vademecum", "calcular_dosis", "derivar"]
PLAN_RESPALDO = [{"id":1,"accion":"clasificar_esi"},{"id":2,"accion":"consultar_protocolo"}]   # mínimo seguro

planner_prompt = ChatPromptTemplate.from_messages([
 ("system", "Eres el planificador de un asistente clínico de SAPU. Descompón el caso en pasos usando SOLO estas acciones: "
  "clasificar_esi, consultar_protocolo, consultar_vademecum, calcular_dosis, derivar. "
  "Reglas: siempre parte con clasificar_esi; calcular_dosis exige consultar_vademecum antes; "
  "incluye derivar solo si el caso podría requerir traslado. "
  "Responde ÚNICAMENTE un JSON: {{\"steps\": [{{\"id\": 1, \"accion\": \"...\"}}]}}"),
 ("human", "Caso: {caso}")])

def extraer_json(texto):
    limpio = re.sub(r"^\s*```(?:json)?\s*|\s*```\s*$", "", str(texto).strip(), flags=re.I)
    i, j = limpio.find("{"), limpio.rfind("}")
    return json.loads(limpio[i:j+1])

def validar_plan(obj):
    pasos = obj.get("steps")
    assert isinstance(pasos, list) and 1 <= len(pasos) <= 8, "plan vacío o demasiado largo"
    vistas = []
    for p in pasos:
        assert p.get("accion") in ACCIONES, f"acción no permitida: {p.get('accion')}"
        if p["accion"] == "calcular_dosis": assert "consultar_vademecum" in vistas, "calcular_dosis sin vademécum previo"
        if p["accion"] == "derivar": assert "clasificar_esi" in vistas, "derivar sin clasificación ESI previa"
        vistas.append(p["accion"])
    return [{"id": i+1, "accion": p["accion"]} for i, p in enumerate(pasos)]

def planificar(caso_texto):
    try:
        out = (planner_prompt | llm).invoke({"caso": caso_texto}).content
        plan = validar_plan(extraer_json(out)); origen = "LLM"
    except Exception as e:
        print(f"⚠️ Plan del LLM rechazado ({e}). Se usa plan de respaldo.")
        plan, origen = [dict(p) for p in PLAN_RESPALDO], "RESPALDO"
    print(f"Plan ({origen}):", [p["accion"] for p in plan])
    return plan

## 6. Ejecutor con estado, priorización y reglas adaptativas

In [ ]:
class DependenciaError(Exception): pass

ORDEN = {"clasificar_esi":0, "derivar":1, "consultar_protocolo":2, "consultar_vademecum":3, "calcular_dosis":4}  # prioridad
NUM = r"(\d+(?:[.,]\d+)?)"

# ---------- Handlers: cada acción usa las herramientas de IL2.1 y el estado compartido ----------
def h_clasificar_esi(e):
    c = e["caso"]   # Reglas DEMO: reemplazar por criterios reales del Manual ESI
    e["esi"] = 1 if c.get("riesgo_vital") else 2 if c.get("alto_riesgo") else 3
    return f"ESI {e['esi']} (reglas DEMO)"

def h_consultar_protocolo(e):
    r = buscar_protocolo.invoke({"consulta": e["caso"]["consulta_protocolo"]})
    e["resultados"]["protocolo"] = r
    return r.split("\n")[0][:160]

def h_consultar_vademecum(e):
    r = consultar_vademecum.invoke({"farmaco": e["caso"]["farmaco"]})
    mg = re.search(NUM + r"\s*mg/kg", r); mx = re.search(r"m[aá]ximo\s*" + NUM + r"\s*mg", r)
    if "NO figura" in r or not (mg and mx):
        e["dosis_ref"] = None; e["vademecum_ok"] = False
        return "Sin dosis utilizable en vademécum"
    e["dosis_ref"] = (float(mg.group(1).replace(",", ".")), float(mx.group(1).replace(",", ".")))
    e["vademecum_ok"] = True
    return f"{e['dosis_ref'][0]:g} mg/kg, máx {e['dosis_ref'][1]:g} mg"

def h_calcular_dosis(e):
    if not e.get("dosis_ref"): raise DependenciaError("no hay dosis de referencia del vademécum")
    mgkg, mx = e["dosis_ref"]
    return calcular_dosis.invoke({"peso_kg": e["caso"]["peso_kg"], "dosis_mg_por_kg": mgkg, "max_mg": mx})

def h_derivar(e):
    if e.get("esi") is None: raise DependenciaError("derivar requiere ESI")
    criterio = e["resultados"].get("protocolo", "Clasificación ESI del sistema")[:200]
    return registrar_derivacion.invoke({"motivo": e["caso"]["motivo"], "criterio_protocolo": criterio, "prioridad_esi": e["esi"]})

HANDLERS = {"clasificar_esi":h_clasificar_esi, "consultar_protocolo":h_consultar_protocolo,
            "consultar_vademecum":h_consultar_vademecum, "calcular_dosis":h_calcular_dosis, "derivar":h_derivar}

# ---------- Reglas adaptativas: se evalúan tras cada paso ----------
def adaptar(e, pendientes, ejecutadas):
    decisiones = []
    c = e["caso"]
    riesgo = c.get("riesgo_vital") or c.get("alto_riesgo")
    if riesgo and e.get("esi_version") != (c.get("riesgo_vital"), c.get("alto_riesgo")):
        e["esi_version"] = (c.get("riesgo_vital"), c.get("alto_riesgo"))
        pendientes.insert(0, {"accion": "clasificar_esi"}); decisiones.append("Condición cambió → re-clasificar ESI")
    if riesgo and not any(p["accion"] == "derivar" for p in pendientes + ejecutadas):
        pendientes.append({"accion": "derivar"}); decisiones.append("Riesgo alto → agregar derivación al plan")
    if c.get("riesgo_vital"):
        antes = len(pendientes)
        pendientes[:] = [p for p in pendientes if p["accion"] not in ("consultar_vademecum", "calcular_dosis")]
        if len(pendientes) < antes: decisiones.append("Riesgo vital → se difiere dosificación; prioridad = derivar")
    if e.get("vademecum_ok") is False and any(p["accion"] == "calcular_dosis" for p in pendientes):
        pendientes[:] = [p for p in pendientes if p["accion"] != "calcular_dosis"]
        e["escalar_medico"] = True; decisiones.append("Sin dosis en vademécum → se omite cálculo y se escala al médico")
    return decisiones

def ejecutar_plan(plan, caso, eventos=None):
    e = {"caso": dict(caso), "esi": None, "dosis_ref": None, "resultados": {}}
    pendientes, ejecutadas, traza, n = [dict(p) for p in plan], [], [], 0
    while pendientes:
        pendientes.sort(key=lambda p: ORDEN[p["accion"]])          # priorización
        paso = pendientes.pop(0); n += 1
        try:
            salida, estado_paso = HANDLERS[paso["accion"]](e), "ok"
        except DependenciaError as ex:
            salida, estado_paso = f"omitido: {ex}", "omitido"; e["escalar_medico"] = True
        except Exception as ex:
            salida, estado_paso = f"error: {ex}", "error"; e["escalar_medico"] = True
        ejecutadas.append(paso)
        decisiones = []
        if eventos and n in eventos:                                  # evento externo simulado
            e["caso"].update(eventos[n]["cambios"]); decisiones.append("EVENTO: " + eventos[n]["descripcion"])
        decisiones += adaptar(e, pendientes, ejecutadas)
        traza.append({"n": n, "accion": paso["accion"], "estado": estado_paso, "salida": str(salida)[:110], "decisiones": decisiones})
    return e, traza

def mostrar(e, traza):
    for t in traza:
        print(f"[{t['n']}] {t['accion']:<20} {t['estado']:<8} {t['salida']}")
        for d in t["decisiones"]: print(f"      ↳ {d}")
    print("ESI final:", e["esi"], "| Escalar a médico:", bool(e.get("escalar_medico")))

## 7. Pruebas de planificación y decisiones adaptativas (evidencia IE5 / IE7)

In [ ]:
CASO = {"motivo": "Crisis asmática moderada pediátrica de 15 kg",
        "consulta_protocolo": "crisis asmática moderada pediátrica",
        "farmaco": "hidrocortisona", "peso_kg": 15, "riesgo_vital": False, "alto_riesgo": False}
TEXTO = "Paciente pediátrico de 15 kg con crisis asmática moderada. Evaluar protocolo y dosis de hidrocortisona."

# E1 — flujo normal: plan completo por prioridad, sin derivación
plan = planificar(TEXTO)
e, traza = ejecutar_plan(plan, CASO); mostrar(e, traza)

Plan (LLM): ['clasificar_esi', 'consultar_protocolo', 'consultar_vademecum', 'calcular_dosis']
[1] clasificar_esi       ok       ESI 3 (reglas DEMO)
[2] consultar_protocolo  ok       [Guía Clínica Asma <15 años (DEMO), p.12] DEMO. Crisis asmática moderada pediátrica: administrar salbutamol in
[3] consultar_vademecum  ok       5 mg/kg, máx 100 mg
[4] calcular_dosis       ok       Dosis calculada: 75 mg. Requiere validación del médico a cargo.
ESI final: 3 | Escalar a médico: False


In [ ]:
# E2 — CAMBIO DE CONDICIONES: tras el paso 2 el paciente empeora → el agente re-planifica
plan = planificar(TEXTO)
ev = {2: {"descripcion": "paciente empeora (sin respuesta a tratamiento)",
          "cambios": {"riesgo_vital": True, "motivo": "Crisis asmática con deterioro: sin respuesta a tratamiento"}}}
e, traza = ejecutar_plan(plan, CASO, eventos=ev); mostrar(e, traza)

Plan (LLM): ['clasificar_esi', 'consultar_protocolo', 'consultar_vademecum', 'calcular_dosis']
[1] clasificar_esi       ok       ESI 3 (reglas DEMO)
[2] consultar_protocolo  ok       [Guía Clínica Asma <15 años (DEMO), p.12] DEMO. Crisis asmática moderada pediátrica: administrar salbutamol in
      ↳ EVENTO: paciente empeora (sin respuesta a tratamiento)
      ↳ Condición cambió → re-clasificar ESI
      ↳ Riesgo alto → agregar derivación al plan
      ↳ Riesgo vital → se difiere dosificación; prioridad = derivar
[3] clasificar_esi       ok       ESI 1 (reglas DEMO)
[4] derivar              ok       Ficha guardada en derivaciones/derivacion_20261028_150316.json (pendiente de validación médica).
ESI final: 1 | Escalar a médico: False


In [ ]:
# E3 — INFORMACIÓN FALTANTE: fármaco que no existe en el vademécum → omite dosis y escala al médico
caso3 = dict(CASO, farmaco="zorbalix", motivo="Consulta por fármaco no disponible")
plan = planificar("Paciente de 15 kg. Calcular dosis de zorbalix según vademécum.")
e, traza = ejecutar_plan(plan, caso3); mostrar(e, traza)

Plan (LLM): ['clasificar_esi', 'consultar_vademecum', 'calcular_dosis']
[1] clasificar_esi       ok       ESI 3 (reglas DEMO)
[2] consultar_vademecum  ok       Sin dosis utilizable en vademécum
      ↳ Sin dosis en vademécum → se omite cálculo y se escala al médico
ESI final: 3 | Escalar a médico: True


In [ ]:
# E4 — VALIDACIÓN DEL PLAN: un plan inválido (acción inventada / dosis sin vademécum) es rechazado
for malo in ['{"steps":[{"accion":"recetar_antibiotico"}]}',
             '{"steps":[{"accion":"clasificar_esi"},{"accion":"calcular_dosis"}]}']:
    try: validar_plan(extraer_json(malo))
    except AssertionError as ex: print("Rechazado →", ex)

Rechazado → acción no permitida: recetar_antibiotico
Rechazado → calcular_dosis sin vademécum previo
